<a href="https://colab.research.google.com/github/nikhilkanse2596/Library-Management-System/blob/main/Library_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re

FILENAME = 'Library_data'
FILEPATH = 'drive/MyDrive/Colab Notebooks/DY CLASSES/Library_data.txt'
DATAFILE = 'drive/MyDrive/Colab Notebooks/DY CLASSES/Login_data.txt'
DONATEDFILE = 'drive/MyDrive/Colab Notebooks/DY CLASSES/Donated_books.txt'
BORROWBOOK = 'drive/MyDrive/Colab Notebooks/DY CLASSES/borrow_books.txt'

SEPARATOR = "|"


class Book:
    def __init__(self,title,author,code):
        self.title = title
        self.author = author
        self.code = code

    def data_format(self):
        return f"{self.title}{SEPARATOR}{self.author}{SEPARATOR}{self.code}\n"


class Donated_data:
    def __init__(self,title,author):
        self.title = title
        self.author = author

    def data_format(self):
        return f"{self.title}{SEPARATOR}{self.author}\n"

class Borrowed_data:
    def __init__(self,sid,title,author,code):
        self.title = title
        self.author = author
        self.sid = sid
        self.code = code

    def data_format(self):
        return f"{self.sid}{SEPARATOR}{self.title}{SEPARATOR}{self.author}{SEPARATOR}{self.code}\n"

#list of dict. to save individual login creds

class student_registration:
    def __init__(self):
        self.load_student_cred()


    def reg_format(self):
        return f"{self.fname}{SEPARATOR}{self.lname}{SEPARATOR}{self.sid}{SEPARATOR}{self.password}\n"

    def load_student_cred(self):
        self.log_cred = []

        if os.path.exists(DATAFILE):
            try:
                with open(DATAFILE, 'r') as file:
                    for line in file:
                        cln_cred = line.strip()
                        if cln_cred:
                            parts = cln_cred.split(SEPARATOR)
                            if len(parts) == 4:
                                indi_cred = {
                                    'fname': parts[0],
                                    "lname": parts[1],
                                    "sid": parts[2],
                                    "password": parts[3]
                                }
                                self.log_cred.append(indi_cred)

            except Exception as e:
                print('Error: ',e)
        else:
            print('‼️NO LOGIN DATA FOUND')

    def get_log_cred(self):
        return self.log_cred


    def validate_cred(self):

        id = get_valid_input('Enter Your UserID / Email ID: ', valid_type='username')
        password = get_valid_input('Enter Your Password: ', valid_type='userpass')
        for i, line in enumerate(self.log_cred):
            if id in self.log_cred[i]['sid'] and password in self.log_cred[i]['password']:
                print()
                print('=='*25)
                print(f"           WELCOME!  {self.log_cred[i]['fname']} {self.log_cred[i]['lname']}")
                print('=='*25)
                student_menu()
                return
        else:
            print('\n❌Wrong Credentails. Check for correct Credentials')
            print('‼️NEW STUDENT REGISTER FIRST\n')


    def register(self,fname,lname,sid,password):
        self.fname = fname
        self.lname =lname
        self.sid = sid
        self.password = password
        with open(DATAFILE, 'a') as file:
            file.write(self.reg_format())
        print("✅ STUDENT DETAILS ADDED SUCCESSFULLY")
        print()
        print("🫡 NOW YOU CAN GO TO STUDENT LOGIN\n")


class Library:
    def __init__(self):
        self.load_books()
        self.load_donated_list()
        self.load_borrowed_list()
        self.var = student_registration()
        # log_cred


    def load_books(self):
        self.books = []
        if os.path.exists(FILEPATH):
            try:
                with open(FILEPATH, 'r') as file:
                    for line in file:
                        cln_line = line.strip()
                        if cln_line:
                            parts = cln_line.split(SEPARATOR)
                            if len(parts) == 3:
                                book_data = {
                                    'title': parts[0],
                                    'author': parts[1],
                                    'code': parts[2]
                                }
                                self.books.append(book_data)
                print(f"\nData loaded successfully from {FILENAME}: Library has {len(self.books)} books in it.")

            except Exception as e:
                print(f"🥱 Error while loading file: {e}. Creating a new Library")
        else:
            print(f"🥱 Library doesnt exists. Creating a new Library file.")


    def load_donated_list(self):
        self.donated_books = []
        if os.path.exists(DONATEDFILE):
            try:
                with open(DONATEDFILE, 'r') as file:
                    for line in file:
                        cln_line = line.strip()
                        if cln_line:
                            parts = cln_line.split(SEPARATOR)
                            if len(parts) == 2:

                                book_data = {
                                    'title': parts[0],
                                    'author': parts[1]
                                }
                                self.donated_books.append(book_data)

            except Exception as e:
                print(f"🥱 Error while loading file: {e}. Creating a new File")
        else:
            print(f"🥱 Donated list doesnt exists. Creating a new Library file.")


    def load_borrowed_list(self):
        self.borrowed_books = []
        if os.path.exists(BORROWBOOK):
            try:
                with open(BORROWBOOK, 'r') as file:
                    for line in file:
                        cln_line = line.strip()
                        if cln_line:
                            parts = cln_line.split(SEPARATOR)
                            if len(parts) == 4:

                                book_data = {
                                    'sid': parts[0],
                                    'title': parts[1],
                                    'author': parts[2],
                                    'code': parts[3]
                                }
                                self.borrowed_books.append(book_data)

            except Exception as e:
                print(f"🥱 Error while loading file: {e}. Creating a new File")
        else:
            print(f"🥱 Borrowed_books file doesnt exists. Creating a new Library file.")


    def save_borrowed_books(self):
        with open(BORROWBOOK, 'w') as file:
            for line in self.borrowed_books:
                temp_data = Borrowed_data(line['sid'], line['title'], line['author'], line['code'])
                file.write(temp_data.data_format())
            print(f"\n✅ Books is added to Borrowed list!")


    def borrow_books(self):
        self.log_cred = self.var.get_log_cred()
        self.list_books()

        if not self.books:
            return

        if self.books:
            book_code = get_valid_input('Enter Book Code to Borrow (or 0 to exit): ', valid_type='code')
            if book_code == '0':
                print("‼️ Exited By User")
                return

            for i,line in enumerate(self.books):
                if self.books[i]['code'] == book_code:
                    sid = get_valid_input('Enter Your UserID / Email ID: ', valid_type='username')
                    for data in self.log_cred:
                        if sid in data['sid']:
                            book_data = {
                                    'sid': data["sid"],
                                    'title': self.books[i]['title'],
                                    'author': self.books[i]['author'],
                                    'code': self.books[i]['code']
                                    }
                            self.borrowed_books.append(book_data)
                            self.save_borrowed_books()
                            return
            else:
                print("Wrong Book Code")

        else:
            print("💢 library doesn't consist any Book to Borrow")


    def save_books(self):
        with open(FILEPATH, 'w') as file:
            for line in self.books:
                temp_data = Book(line['title'],line['author'],line['code'])
                file.write(temp_data.data_format())
        print(f"✅ Data saved successfully to {FILENAME}")
        return


    def add_book(self,obj):
        book_data = {
            "title": obj.title,
            'author': obj.author,
            'code': obj.code
        }
        self.books.append(book_data)
        print(f"\n✅ Book with Title: {obj.title} is added to Library!")
        self.save_books()
        # print("‼️ Choose option 5 to save the changes to the file.")

    def donate_book(self,obj):
        with open(DONATEDFILE, 'a') as file:
            temp_data = Donated_data(obj.title, obj.author)
            file.write(temp_data.data_format())
        print(f"\n✅ Book with Title: {obj.title} is added to Donated list! Admin will check and approve")
        # print("‼️ Choose option 5 to save the changes to the file.")


    def list_books(self):
        if not self.books:
            print(f"\n🤷‍♀️😶 OH-HO. {FILENAME} is empty. Add data of Books by choosing option 1")


        if self.books:
            print("----------------| 📚 All Books inside our Library 📚 |----------------\n")
            for i,data in enumerate(self.books):
                print(f"{i+1:2d}. Title: {data['title']:30s} | Author: {data['author']:30s} | Book Code: {data['code']}\n")
            print("________________________THE END (of book list)_________________________\n")
            return


    def search_book(self):
        results = []

        if self.books:
            search = input("Enter Title, Author Name or Book Code to search (or 0 exit): ").strip()
            if not search:
                print("\n🙃 Input cannot be empty. try again.")
                return

            if search == "0":
                return

            if search:
                for line in self.books:
                    if search.lower() in line['title'].lower() or \
                    search.lower() in line['author'].lower() or \
                    search == line['code']:
                        results.append(line)
        else:
            print("🙃 Library is empty to search anything. first add books")

        if results:
            print(f"\n\n------------------Search Result for: [{search}]------------------\n")
            for line in results:
                    print(f"Title: {line['title']:30s} | Author: {line['author']:30s} | Book Code: {line['code']}")
            print("\n________________________THE END (of Search list)_________________________\n")
        else:
            print(f"\n----- ❌ No Result found for [{search}]-----\n")


    def Donated_list(self):
        if not self.donated_books:
            print(f"\n🤷‍♀️😶 Donated file is empty.")
            return

        if self.donated_books:


                print("----------------| 📚 All Books inside our DONATED BOOKS FILE 📚 |----------------\n")
                for i,data in enumerate(self.donated_books):
                    print(f"{i+1:2d}. Title: {data['title']:15s} | Author: {data['author']}\n")
                print("________________________THE END (of book list)_________________________\n")
                return


    def Borrowed_list(self):
        if not self.borrowed_books:
            print(f"\n🤷‍♀️😶 Borrowed file is empty.")
            return

        if self.borrowed_books:
            print("----------------| 📚 All Books inside our BORROWED BOOKS FILE 📚 |----------------\n")
            for i,data in enumerate(self.borrowed_books):
                print(f"{i+1:2d}. Title: {data['title']:15s} | Author: {data['author']} | Book Code: {data['code']}\n")
            print("________________________THE END (of book list)_________________________\n")
            return



    def approve_book(self):
        if not self.donated_books:
            return

        try:
            task_num = int(input("Enter the number of the Book to Approve (or 0 to cancel): "))
            if task_num == 0:
                print("‼️ Operation Canceled by ADMIN")
                print('-'*60)
                return

            if 1 <= task_num <= len(self.donated_books):
                mark_index = task_num - 1
                book_code = get_valid_input('Enter Book Code for selected book: ', valid_type='code')
                self.donated_books[mark_index]['code'] = book_code

                for i, line in enumerate(self.donated_books):
                    if len(self.donated_books[i]) == 3:
                        self.books.append(self.donated_books[i])
                        del self.donated_books[i]
                        with open(DONATEDFILE, 'w') as file:
                            for line in self.donated_books:
                                temp_data = Donated_data(line['title'],line['author'])
                                file.write(temp_data.data_format())
                else:
                    print("✅ BOOK APROVED SUCCESSFULLY")

            else:
                print(f'\n😒 Invalid Input. Enter number between 1 and {len(self.donated_books)}.')
                print('-'*70)

        except ValueError:
            print("\nError: 😒 Invalid input. Enter a number.")
            print('-'*70)



    def delete_book(self,num):
        ini_cnt = len(self.books)

        self.books = [
            book_data
            for book_data in self.books
            if book_data['code'] != num
        ]

        del_books = ini_cnt - len(self.books)

        if del_books > 0:
            print(f"\n✅ Successfully deleted {del_books} book with Book Code: [{num}]")
            print("‼️ Choose option 5 to save the changes to the file.")
        else:
            print(f"\n🗿 No book found with Book Code: [{num}]. Deleting failed.")




#this below code is to take some restricted input only. with condition
def get_valid_input(string, valid_type='title_author'):
    while True:
        user_input = input(string).strip()
        if not user_input:
            print("🙃 Input cannot be empty. try again.")
            continue

        if valid_type == 'code':
            # strict alphanumeric for code (no spaces allowed)
            if not re.fullmatch(r"^[a-zA-Z0-9-]+$", user_input):
                print("🚫 Book Code must be alphanumeric (letters or numbers or dash[-] only, no spaces or other symbols). Try again.")
                continue

        elif valid_type == 'title_author':
            # allowing letters, numbers, spaces, ('), (-), commas (,), (.).
            if not re.fullmatch(r"^[a-zA-Z0-9\s',\.-]+$", user_input):
                print("🚫 Input can contain only letters, numbers, spaces, and (' , . -). Try again.")
                continue

        elif valid_type == 'user-fl-name':
            # strict alphabetic
            if not user_input.isalpha():
                print("🚫 First Name or Last name should only contain alphabets (a-z A-Z). Try again.")
                continue

        elif valid_type == 'username':
            # allowing letters, numbers and @.
            if not re.fullmatch(r"^[a-zA-Z0-9._-]+@gmail\.com$", user_input):
                print("🚫 Enter Valid Email Id / UserId. Standard format-(abc@gmail.com).")
                print('📃 Id can contain (a-z A-Z 0-9 "." "_") and should follow standard Email format. Try again')
                continue

        elif valid_type == 'userpass':

            if not re.fullmatch(r"^[a-zA-Z0-9@]+$", user_input):
                print("🚫 Input can contain only letters, numbers, and (@). Try again.")
                continue

        return user_input




# now we will write code to execute the classes(options to chosoe)
def ad_menu():
    print("\n\n______📚 Library Management System Menu 📚______")
    print("1. Add New Book")
    print("2. List All Books")
    print("3. Search Book (by Title, Author or Book Code)")
    print("4. Approve Donated books")
    print("5. Remove Book from Library (by Code)")
    print("6. View Borrowed Books list")
    print("7. Save and Exit")
    print('_'*50, "\n")

def st_menu():
    print("\n\n______📚 Library Management System Menu 📚______")
    print("1. Donate Book")
    print("2. List All Books")
    print("3. Search Book (by Title, Author or Book Code)")
    print("4. Borrow book")
    print("5. View Borrowed book")
    print("6. Save and Exit")
    print('_'*50, "\n")

def admin_menu():
    my_library = Library()

    while True:
        ad_menu()
        choice = input('Enter Your Choice (1-6): ')
        if choice == '1':
            print('\n‼️Note: Title/Author can contain spaces. Code is strictly alphanumeric.')
            title = get_valid_input('Enter Book Title (or 0 to exit): ', valid_type='title_author')
            if title != '0':
                author = get_valid_input('Enter Author Name: ', valid_type='title_author')
                code = get_valid_input('Enter Book Code: ', valid_type='code')
                new_book = Book(title,author,code)
                my_library.add_book(new_book)


        elif choice == '2':
            my_library.list_books()

        elif choice == '3':
            my_library.search_book()

        elif choice == '4':
            my_library.Donated_list()
            my_library.approve_book()
            my_library.save_books()

        elif choice == '5':
            my_library.list_books()
            del_code = input("Enter *Book Code* to DELETE from Library (or 0 to exit): ")
            if del_code == '0':
                continue

            my_library.delete_book(del_code)

        elif choice == '6':
            my_library.Borrowed_list()

        elif choice == '7':
            my_library.save_books()
            print("🙏Thanks for using Library Management System by THE NIKHIL KANSE.\nGoodbye!\nHave a nice day!")
            break

        else:
            print('🙃 Invalid choice. Enter a number between 1 and 5.')


#for student interface
def student_menu():
    my_library = Library()

    while True:
        st_menu()
        choice = input('Enter Your Choice (1-6): ')
        if choice == '1':
            print('\n‼️Note: Title/Author can contain spaces.')
            title = get_valid_input('Enter Book Title (or 0 to exit): ', valid_type='title_author')
            if title == '0':
                print("‼️ Exited by User")
                return
            author = get_valid_input('Enter Author Name: ', valid_type='title_author')
            new_book = Donated_data(title,author)
            my_library.donate_book(new_book)

        elif choice == '2':
            my_library.list_books()

        elif choice == '3':
            my_library.search_book()

        elif choice == '4':
            my_library.borrow_books()
            # my_library.save_borrowed_books()

        elif choice == '5':
            my_library.Borrowed_list()

        elif choice == '6':
            my_library.save_books()
            print("🙏Thanks for using Library Management System by THE NIKHIL KANSE.\nGoodbye!\nHave a nice day!")
            break

        else:
            print('🙃 Invalid choice. Enter a number between 1 and 5.')




def student_section():
    fname = get_valid_input('Enter Your First Name (or z to exit): ', valid_type='user-fl-name').strip().capitalize()
    if fname == 'Z':
        print('‼️ User exited from Program')
        return
    lname = get_valid_input('Enter Your Last Name: ', valid_type='user-fl-name').strip().capitalize()
    sid = get_valid_input('Enter Your UserID / Email ID: ', valid_type='username')
    password = get_valid_input('Enter Your Password: ', valid_type='userpass')
    my_details = student_registration()
    my_details.register(fname, lname, sid, password)



def login_page():
    print("\n______📚 Library Management System Menu 📚______")
    print()
    print('\n             Select option to Login')
    print('_'*50, "\n")
    print("1. ADMIN LOGIN")
    print("2. STUDENT LOGIN")
    print("3. REGISTER NEW STUDENT")
    print("4. EXIT")
    print('_'*50, "\n")


# login of ADMIN
def login_admin():
    print("\n\n______📚 Library Management System Menu 📚______")
    print()
    print('               Login to Continue')
    print()

    ID = input("Enter Login ID: ")
    Pass = input("Enter Login Password: ")


    if ID == 'nikhilkanse@admin.com' and Pass == 'Admin@1234':
        print('\n\n')
        print('=='*25)
        print("              Welcome! Nikhil Kanse")
        print('__'*25)

        admin_menu()
        # creating the library obj
        # my_library = Library()

    else:
        print('‼️ Invalid Login ID or Password')


while True:
    login_page()

    choice = input('Enter your choice (1-4): ')

    if choice == '1':
        login_admin()
        break

    elif choice == '2':
        getin = student_registration()
        getin.validate_cred()


    elif choice == '3':
        student_section()

    elif choice == '4':
        print('‼️ User Exited Program')
        print('🙏Thanks for using Library Management System by THE NIKHIL KANSE.\nGoodbye!\nHave a nice day!')
        break

    else:
        print('‼️Invalid input. Enter 1-4')



______📚 Library Management System Menu 📚______


             Select option to Login
__________________________________________________ 

1. ADMIN LOGIN
2. STUDENT LOGIN
3. REGISTER NEW STUDENT
4. EXIT
__________________________________________________ 

Enter your choice (1-4): 4
‼️ User Exited Program
🙏Thanks for using Library Management System by THE NIKHIL KANSE.
Goodbye!
Have a nice day!
